In [ ]:
import param
import panel as pn
import numpy as np
import holoviews as hv
from functools import partial

pn.extension()

## Baseclasses

These baseclasses were taken straight from another dashboard project and give an overall structure to the dashboard which allows opening multiple dashboard tabs.

In [ ]:
class Application(param.Parameterized):
    """
    The Application class is the top-level object which coordinates
    the layout of the different components of the dashboard. It
    primarily consists of a header and body, the header is generated
    automatically from the current title and a logo while the body
    is a :class:`Component` which implements a :meth:`Component.panel`
    method which returns panel objects which can be composed into the
    overall application layout.
    """

    body = param.ClassSelector(class_=param.Parameterized, doc="""
        The Component to render in the application's body.""")

    logo = param.String('https://www.lsst.org/sites/default/files/logos/LSST_web_white.png', doc="""
        The logo to display in the header.""")

    title = param.String(doc="The current title to display in the header")

    def __init__(self, **params):
        super().__init__(**params)
        self._title = pn.pane.HTML(
            '<h1>%s</h1>' % self.title, margin=(40, 80), height=100,
            sizing_mode='stretch_width', style={'white-space': 'nowrap'}
        )
        logo = pn.pane.PNG(self.logo, width=400, height=150)
        self.header = pn.Row(self._title, pn.layout.HSpacer(max_height=0),
                             logo, margin=0, max_height=200)

    @param.depends('title', watch=True)
    def _update_title(self):
        "Upddates the title "
        self._title.object = '<h1>%s</h1>' % self.title

    @param.depends('body')
    def get_body(self):
        self.body.application = self
        return self.body.panel()

    def render(self):
        "Renders the application as a single panel layout."
        self.body.application = self
        return pn.Column(self.header, self.get_body(),
                         width_policy='max', height_policy='max')

    
class Component(param.Parameterized):
    """
    Baseclass for visual components of the :class:`Application`.
    Defines a common API shared by all displayable components which
    allows access to the parent object, the :class:`Application` and
    defines methods which should return the title and a displayable
    panel object.
    """

    application = param.ClassSelector(class_=Application, precedence=-1, doc="""
        The :class:`Application` responsible for rendering the component.""")

    parent = param.ClassSelector(class_=param.Parameterized, precedence=-1, doc="""
        The parent :class:`Component` of the the object.""")

    label = param.String(default='Component', precedence=-1, doc="""
        A label to identify the Component by.""")

    __abstract = True

    def title(self):
        """
        Title to be displayed as part of the Application header.
        """
        return type(self).__name__

    def panel(self):
        """
        Should return a panel object to be rendered as part of the
        application.
        """
        
class TabComponent(Component):
    """
    The TabComponent displays a set of Tabs which may each contain a
    different :class:`Component`. This allows moving back and forth
    between stages of the :class:`Application`.
    """

    current = param.ClassSelector(class_=Component, precedence=-1, doc="""
        The component to display in the current tab..""")

    objects = param.List(doc="The list of sub-components to render in the tabs.")

    def __init__(self, *objects, **params):
        super().__init__(objects=list(objects), **params)
        self._layout = pn.Tabs(sizing_mode='stretch_both')
        self._layout.param.watch(self._update_title, 'active')

    def _update_title(self, event):
        "Updates the Application title depending on the selected tab."
        self.application.title = '<h1>%s</h1>' % self.objects[event.new].title()

    @param.depends('application', watch=True)
    def _init_panel(self):
        "Initializes the layout when the Application is initialized."
        for obj in self.objects:
            obj.parent = self
            obj.application = self.application
        self._layout[:] = [(obj.label, obj.panel()) for obj in self.objects]
        if self.current is None and self.objects:
            self.current = self.objects[0]

    @param.depends('current', watch=True)
    def _update_view(self):
        "Updates the current view whenever the current parameter is set."
        self.current.application = self.application
        obj_types = [type(obj) for obj in self.objects]
        title = self.current.title()
        if type(self.current) in obj_types:
            idx = obj_types.index(type(self.current))
            self.objects[idx] = self.current
            self._layout[idx] = (self.current.label, self.current.panel())
        else:
            self.objects.append(self.current)
            self._layout.append((self.current.label, self.current.panel()))
        self._layout.active = self.objects.index(self.current)
        self.application.title = title

    def panel(self):
        self._init_panel()
        return self._layout

## LSST specific classes

These LSST specific components plug into the dashboard architecture and fill in the actual content of the app.

In [ ]:
class QuickLookComponent(Component):
    
    data_repository = param.String()
    
    comparison = param.String()
    
    selected = param.Tuple(default=(None, None, None, None), length=4)
    
    label = param.String(default='Quick Look')

    def __init__(self, **param):
        super().__init__(**param)
        self._submit_repository = pn.widgets.Button(
            name='Submit', width=50, align='end')
        self._submit_comparison = pn.widgets.Button(
            name='Submit', width=50, align='end')
        self._open_detailed = pn.widgets.Button(
            name='Open Detailed View', disabled=any(v is None for v in self.selected))
        self._submit_repository.on_click(self._update)
        self._submit_comparison.on_click(self._update)
        self._open_detailed.on_click(self._open_detailed_view)
        self._info = pn.pane.HTML(width=600)
        self._metric_panels = []
        self._metric_layout = pn.Column()
        self._selected_info = pn.pane.HTML(width=300)
        self._update(None)

    def title(self):
        return 'LSST Data Processing Explorer - Quick Look'
    
    def _update(self, event):
        self._update_info()
        self._load_metrics()
    
    def _update_info(self):
        """
        Updates the _info HTML pane with info loaded from the current repository.
        """
        html = """
        <code>
        Tracts: 8, patches = 648<br>
        Visits: 129<br>
        Filters (visits): HSC-G (25), HSG-R (24), HSC-I (22), HSC-Y (30), HSC-Z (28)<br>
        Unique Objects: 8,250,442
        </code>
        """
        self._info.object = html
        
    def _open_detailed_view(self, event):
        """
        This opens the detail dashboard in a new Tab.
        """
        self.parent.current = DetailComponent(
            parent=self.parent, overview=self)
        
    def _load_metrics(self):
        """
        Populates the _metrics Row with metrics loaded from the repository
        """
        # Load filters from repository
        filters = ['HSC-G', 'HSG-R', 'HSC-I', 'HSC-Y', 'HSC-Z']
        # Load metrics from repository
        metrics = ['Photometry', 'Astrometry', 'Shape', 'Color']
        panels = [MetricPanel(metric=metric, filters=filters, parent=self) for metric in metrics]
        self._metric_panels = panels
        self._metric_layout.objects = [p.panel() for p in panels]
        
    @param.depends('selected', watch=True)
    def _update_selected(self):
        self._selected_info.object = (
            "Selected metric: %s<br> Filter: %s<br> Clicked on: (%s, %s)" % self.selected)
        self._open_detailed.disabled = False

    def panel(self):
        row1 = pn.Row(self.param.data_repository, self._submit_repository)
        row2 = pn.Row(self.param.comparison, self._submit_comparison)
        return pn.Column(
            pn.Row(
                pn.Column(row1, row2),
                pn.layout.HSpacer(),
                self._info
            ),
            pn.pane.HTML('<hr width=100%>', sizing_mode='stretch_width'),
            pn.Row(
                self._metric_layout,
                pn.Column(
                    self._selected_info,
                    self._open_detailed
                )
            ),
            sizing_mode='stretch_both'
        )
    
class MetricPanel(param.Parameterized):
    """
    A MetricPanel displays clickable heatmaps for a particular metric,
    broken down into separate tabs for each filter.
    """
    
    metric = param.String()

    parent = param.ClassSelector(class_=QuickLookComponent)
    
    filters = param.List()
    
    def __init__(self, **params):
        super().__init__(**params)
        self._streams = []
        self._heatmaps = [(filt, self._create_metric_heatmap(filt)) for filt in self.filters]

    def _create_metric_heatmap(self, filt):
        """
        Access repository from parent and populate heatmap
        """
        heatmap = hv.HeatMap((range(5), range(2), np.random.randint(0, 2, (2, 5)))).opts(
            height=100, cmap=['red', 'green'], xaxis=None, yaxis=None, toolbar=None,
            line_alpha=1, line_width=5, line_color='white', width=500, nonselection_fill_alpha=0.2)
        tap = hv.streams.Tap(source=heatmap)
        self._streams.append(tap)
        tap.param.watch(partial(self._tapped, filt), ['x', 'y'])
        return heatmap
    
    def _tapped(self, filt, *args):
        """
        This method is called with information about the metric that was clicked.
        """
        self.parent.selected = (self.metric, filt, args[0].obj.x, args[0].obj.y)

    def panel(self):
        return pn.Column(
            pn.pane.Markdown('### %s Metrics' % self.metric, margin=0),
            pn.Tabs(*self._heatmaps, sizing_mode='stretch_width', margin=0),
            sizing_mode='stretch_width'
        )
    
    
class DetailComponent(Component):
    """
    This is a mockup of the detailed dashboard.
    """
    
    overview = param.ClassSelector(class_=QuickLookComponent)

    label = param.String(default='Detail')
    
    def title(self):
        return 'LSST Data Processing Explorer - Detailed View'
    
    def panel(self):
        return self.overview._selected_info

## Displaying the app

In [ ]:
dashboard = Application(body=TabComponent(QuickLookComponent()))
dashboard.render()